<a href="https://colab.research.google.com/github/carolinampessoa/TechChallengeFase5/blob/main/TechChallengeFase5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1) Importação de bibliotecas**

Este bloco realiza a instalação das duas bibliotecas necessárias para execução do projeto no ambiente Google Colab:

*   OpenAI SDK: utilizada para acessar os modelos de linguagem (LLMs) responsáveis pela análise semântica da arquitetura apresentada na imagem.

*   ReportLab: biblioteca utilizada para geração automática de relatórios em formato PDF.

A instalação via pip garante que o ambiente de execução possua todas as dependências necessárias para executar as etapas de análise, enriquecimento e geração do relatório.

In [ ]:
#Instalação de libs

!pip install openai
!pip install reportlab

**2) Configuração da API da LLM**

Configuração da autenticação necessária para utilizar os modelos de linguagem da OpenAI.

O acesso às APIs da OpenAI exige uma chave de autenticação (API Key).
O usuário informa sua chave manualmente no momento da execução, a qual é armazenada em uma variável de ambiente, chamada OPENAI_API_KEY. Isso permite que o cliente da OpenAI autentique automaticamente todas as chamadas feitas posteriormente no notebook.

Por questão de segurança, usamos a função getpass(), que evita que a chave fique visível no notebook ou registrada no histórico de execução.


In [ ]:
#Configurar API Key (uso de LLM da OpenAI)

import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Digite sua OpenAI API Key: ")

Digite sua OpenAI API Key: ··········


**3) Upload da imagem de arquitetura**

Enviando a imagem a ser analisada.

Como o notebook é executado no Google Colab, não há acesso direto ao computador local do usuário. Portanto:

* A função files.upload() abre uma interface de upload;

* O usuário seleciona a imagem contendo o diagrama arquitetural;

* O arquivo é armazenado temporariamente no ambiente de execução;

O caminho do arquivo é então armazenado na variável image_path, que será utilizada nas etapas seguintes de processamento da imagem.

In [ ]:
#Importar imagem para avaliação

from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]

print("Imagem carregada:", image_path)


Saving Screenshot_1.png to Screenshot_1.png
Imagem carregada: Screenshot_1.png


**4) Funções auxiliares e preparação da imagem**

Preparando os recursos necessários para enviar a imagem ao modelo de linguagem:

* Inicialização do cliente da OpenAI;
* Conversão da imagem: a imagem enviada pelo usuário é convertida para Base64 para que possa ser analisada pelo modelo multimodal da LLM.

In [ ]:
# Funções auxiliares

from openai import OpenAI
import base64
import json

client = OpenAI()

def encode_image(path):
    with open(path, "rb") as img:
        return base64.b64encode(img.read()).decode("utf-8")

base64_image = encode_image(image_path)

**5) Extração livre de componentes da arquitetura**

Identificando automaticamente os elementos arquiteturais presentes no diagrama.
É criado um prompt de análise arquitetural, solicitando ao modelo que examine o diagrama e identifique todos os elementos relevantes.

A instrução enviada ao modelo solicita que ele:

* Analise semanticamente o diagrama;

* Identifique componentes arquiteturais;

* Retorne os resultados em formato JSON estruturado

Essa etapa corresponde à extração automática de conhecimento arquitetural a partir da imagem.

In [ ]:
# Extração Livre de Componentes

free_extraction_prompt = """
Analise o diagrama de arquitetura de software presente na imagem.

Identifique TODOS os elementos arquiteturais semanticamente relevantes.

Não restrinja a categorias pré-definidas.

Responda exclusivamente em JSON válido:

{
  "components": [
    {"name": "...", "type": "..."}
  ]
}
"""

response1 = client.responses.create(
    model="gpt-4.1-mini",
    input=[
        {
            "role": "user",
            "content": [
                {"type": "input_text", "text": free_extraction_prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/png;base64,{base64_image}"
                }
            ]
        }
    ]
)

raw_components_text = response1.output_text
print(raw_components_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "User"},
    {"name": "AWS Shield", "type": "Security Service"},
    {"name": "Amazon CloudFront", "type": "Content Delivery Network"},
    {"name": "AWS WAF", "type": "Web Application Firewall"},
    {"name": "AWS Cloud", "type": "Cloud Infrastructure"},
    {"name": "sa-east-1 (São Paulo)", "type": "AWS Region"},
    {"name": "Virtual Private Cloud", "type": "Network"},
    {"name": "Availability Zone A", "type": "Availability Zone"},
    {"name": "Availability Zone B", "type": "Availability Zone"},
    {"name": "Availability Zone C", "type": "Availability Zone"},
    {"name": "Public Subnet", "type": "Subnet"},
    {"name": "Private Subnet", "type": "Subnet"},
    {"name": "Application Load Balancer", "type": "Load Balancer"},
    {"name": "SEI / SIP (Auto Scaling API Server)", "type": "Compute Instance"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "Storage"},
    {"name": "Amazon RDS (Primary)",

**6) Limpeza e parsing da resposta**

Conversão da resposta da LLM em uma estrutura de dados manipulável pelo Python.
Modelos de linguagem frequentemente retornam JSON envolto em blocos markdown. Este bloco remove esses marcadores e realiza o parsing da resposta.

O processo ocorre em três etapas:

* Remoção de formatação Markdown.
* Conversão do texto JSON para objeto Python.
* Armazenamento na variável `raw_components`.

O resultado final é uma lista estruturada de componentes extraídos do diagrama.

In [ ]:
# Limpeza + Parsing

clean_text = raw_components_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

raw_components = json.loads(clean_text)["components"]

raw_components


[{'name': 'Usuários SEI', 'type': 'User'},
 {'name': 'AWS Shield', 'type': 'Security Service'},
 {'name': 'Amazon CloudFront', 'type': 'Content Delivery Network'},
 {'name': 'AWS WAF', 'type': 'Web Application Firewall'},
 {'name': 'AWS Cloud', 'type': 'Cloud Infrastructure'},
 {'name': 'sa-east-1 (São Paulo)', 'type': 'AWS Region'},
 {'name': 'Virtual Private Cloud', 'type': 'Network'},
 {'name': 'Availability Zone A', 'type': 'Availability Zone'},
 {'name': 'Availability Zone B', 'type': 'Availability Zone'},
 {'name': 'Availability Zone C', 'type': 'Availability Zone'},
 {'name': 'Public Subnet', 'type': 'Subnet'},
 {'name': 'Private Subnet', 'type': 'Subnet'},
 {'name': 'Application Load Balancer', 'type': 'Load Balancer'},
 {'name': 'SEI / SIP (Auto Scaling API Server)', 'type': 'Compute Instance'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ', 'type': 'Storage'},
 {'name': 'Amazon RDS (Primary)', 'type': 'Database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'Database'

**7) Normalização taxonômica dos componentes**

Padronização dos tipos de componentes para uma taxonomia compatível com análise de segurança.
Os componentes identificados pela LLM podem possuir diversos nomes diferentes (EC2 Instance, Backend Server, API Gateway, etc). Para permitir a aplicação consistente do modelo STRIDE, os componentes são normalizados para as seguintes categorias padronizadas:

* user
* server
* database
* api
* external_system

Ou seja, essa etapa transforma descrições semânticas livres em **classes arquiteturais padronizadas**.


In [ ]:
# Normalização Taxonômica

normalization_prompt = f"""
Normalize os componentes abaixo em categorias de segurança.

Categorias permitidas (escolha apenas UMA por item):

- user
- server
- database
- api
- external_system

Componentes:

{json.dumps(raw_components, indent=2)}

Responda em JSON válido:

{{
  "components": [
    {{"name": "...", "type": "..."}}
  ]
}}
"""

response2 = client.responses.create(
    model="gpt-4.1-mini",
    input=normalization_prompt
)

normalized_text = response2.output_text
print(normalized_text)


```json
{
  "components": [
    {"name": "Usuários SEI", "type": "user"},
    {"name": "AWS Shield", "type": "external_system"},
    {"name": "Amazon CloudFront", "type": "external_system"},
    {"name": "AWS WAF", "type": "external_system"},
    {"name": "AWS Cloud", "type": "external_system"},
    {"name": "sa-east-1 (São Paulo)", "type": "external_system"},
    {"name": "Virtual Private Cloud", "type": "external_system"},
    {"name": "Availability Zone A", "type": "external_system"},
    {"name": "Availability Zone B", "type": "external_system"},
    {"name": "Availability Zone C", "type": "external_system"},
    {"name": "Public Subnet", "type": "external_system"},
    {"name": "Private Subnet", "type": "external_system"},
    {"name": "Application Load Balancer", "type": "server"},
    {"name": "SEI / SIP (Auto Scaling API Server)", "type": "api"},
    {"name": "Amazon Elastic File System (NFS) Multi-AZ", "type": "external_system"},
    {"name": "Amazon RDS (Primary)", "type": "d

In [ ]:
# Parsing Normalizado

clean_text = normalized_text.strip()

if clean_text.startswith("```"):
    clean_text = clean_text.replace("```json", "").replace("```", "").strip()

components = json.loads(clean_text)["components"]

components


[{'name': 'Usuários SEI', 'type': 'user'},
 {'name': 'AWS Shield', 'type': 'external_system'},
 {'name': 'Amazon CloudFront', 'type': 'external_system'},
 {'name': 'AWS WAF', 'type': 'external_system'},
 {'name': 'AWS Cloud', 'type': 'external_system'},
 {'name': 'sa-east-1 (São Paulo)', 'type': 'external_system'},
 {'name': 'Virtual Private Cloud', 'type': 'external_system'},
 {'name': 'Availability Zone A', 'type': 'external_system'},
 {'name': 'Availability Zone B', 'type': 'external_system'},
 {'name': 'Availability Zone C', 'type': 'external_system'},
 {'name': 'Public Subnet', 'type': 'external_system'},
 {'name': 'Private Subnet', 'type': 'external_system'},
 {'name': 'Application Load Balancer', 'type': 'server'},
 {'name': 'SEI / SIP (Auto Scaling API Server)', 'type': 'api'},
 {'name': 'Amazon Elastic File System (NFS) Multi-AZ',
  'type': 'external_system'},
 {'name': 'Amazon RDS (Primary)', 'type': 'database'},
 {'name': 'Amazon RDS (Secondary)', 'type': 'database'},
 {'nam

In [ ]:
stride_map = {
    "server": ["Spoofing", "Tampering", "Denial of Service"],
    "database": ["Tampering", "Information Disclosure"],
    "api": ["Spoofing", "Repudiation"],
    "user": ["Spoofing"],
    "external_system": ["Spoofing", "Tampering"]
}

def analyze_stride(components):
    results = []

    for comp in components:
        threats = stride_map.get(comp["type"])
        if threats is None:
            threats = ["Unknown – No STRIDE mapping defined"]

        results.append({
            "component": comp["name"],
            "type": comp["type"],
            "threats": threats
        })

    return results

stride_results = analyze_stride(components)

stride_results


[{'component': 'Usuários SEI', 'type': 'user', 'threats': ['Spoofing']},
 {'component': 'AWS Shield',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Amazon CloudFront',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS WAF',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'AWS Cloud',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'sa-east-1 (São Paulo)',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Virtual Private Cloud',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone A',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone B',
  'type': 'external_system',
  'threats': ['Spoofing', 'Tampering']},
 {'component': 'Availability Zone C',
  'type': 'external_system',
  'threats': ['Spoofing',

In [ ]:
enrichment_prompt = f"""
Considere os resultados de STRIDE abaixo:

{json.dumps(stride_results, indent=2)}

Para cada ameaça, explique o racional técnico e sugira contramedidas.

Resposta em texto estruturado.
"""

response3 = client.responses.create(
    model="gpt-4.1-mini",
    input=enrichment_prompt
)


text_output = response3.output_text
# Remove a última linha da resposta
lines = text_output.strip().split("\n")
text_output = "\n".join(lines[:-1])

print(text_output)


Segue a análise técnica e contramedidas para cada tipo de ameaça identificado no resultado STRIDE fornecido:

---

### 1. Spoofing (Falsificação de identidade)

**Racional técnico:**  
Spoofing ocorre quando um agente malicioso tenta se passar por um usuário, sistema ou componente legítimo, podendo ganhar acesso não autorizado a sistemas ou dados. Isso pode acontecer via falsificação de IP, credentials roubadas, tokens falsificados, entre outros.

**Componentes afetados:** Usuários SEI, AWS Shield, Amazon CloudFront, AWS WAF, AWS Cloud, sa-east-1 (São Paulo), Virtual Private Cloud, Availability Zones A/B/C, Public e Private Subnets, Application Load Balancer, SEI/SIP (Auto Scaling API Server), Amazon Elastic File System, Amazon ElastiCache, AWS CloudTrail, AWS KMS, AWS Backup, Amazon CloudWatch, Amazon SES, Solr (Auto Scaling).

**Contramedidas:**  
- **Autenticação forte:** Implementação de MFA (autenticação multifator) para usuários e sistemas que acessam a infraestrutura.  
- **Cert

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_path = "relatorio_stride.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()

# ===== ESTILOS =====

title_style = ParagraphStyle(
    'Title',
    parent=styles['Title'],
    fontSize=22,
    textColor=colors.darkblue,
    spaceAfter=20
)

section_style = ParagraphStyle(
    'Section',
    parent=styles['Heading2'],
    fontSize=16,
    textColor=colors.darkblue,
    spaceAfter=12
)

subtitle_style = ParagraphStyle(
    'Subtitle',
    parent=styles['Heading3'],
    fontSize=12,
    textColor=colors.black,
    spaceAfter=6
)

body_style = ParagraphStyle(
    'Body',
    parent=styles['BodyText'],
    fontSize=11.5,
    leading=16,
    spaceAfter=8
)

bullet_style = ParagraphStyle(
    'Bullet',
    parent=body_style,
    leftIndent=20,
    spaceAfter=4
)

elements = []

text_output = response3.output_text

elements.append(Paragraph("Relatório de Ameaças – STRIDE", title_style))

for line in text_output.split("\n"):

    line = line.strip()

    if not line:
        elements.append(Spacer(1, 6))
        continue

    # Separadores visuais
    if line.startswith("---"):
        elements.append(Spacer(1, 12))
        continue

    # Títulos principais (###)
    if line.startswith("###"):
        clean = line.replace("###", "").strip()
        elements.append(Spacer(1, 10))
        elements.append(Paragraph(f"<b>{clean}</b>", section_style))
        continue

    # Subtítulos em negrito
    if line.startswith("**") and line.endswith("**"):
        clean = line.replace("**", "")
        elements.append(Paragraph(f"<b>{clean}</b>", subtitle_style))
        continue

    if line.startswith("**"):
        clean = line.replace("**", "").replace(":", "")
        elements.append(Paragraph(f"<b>{clean}:</b>", subtitle_style))
        continue

    # Bullet points
    if line.startswith("-"):
        clean = line[1:].strip()
        elements.append(Paragraph(f"• {clean}", bullet_style))
        continue

    # Texto normal
    elements.append(Paragraph(line, body_style))

doc.build(elements)


In [ ]:
pdf_path = "/content/drive/MyDrive/relatorio_stride.pdf"